<a href="https://colab.research.google.com/github/satvikt09/RAG-assistant-pipeline/blob/main/NLPfinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **NLP Final - RAG based College Assistant**

23BAI1244 - Satvik Tambe

23BAI1021 - Batchu Dinesh Reddy

23BAI1130 - Amaravathi Revanth Kumar

In [ ]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 27.8 MB/s eta 0:00:00


In [ ]:
import textwrap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Load Dataset

In [ ]:
import pandas as pd

df_all = pd.read_pickle("/content/drive/MyDrive/academic_dataset_final.pkl")

print(df_all.shape)

(82880, 3)


Load BERT (Retriever)

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

bert_name = "bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(bert_name)
bert_model = AutoModel.from_pretrained(bert_name).to(device)

bert_model.eval()

print("BERT loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT loaded


Create Embeddings

In [ ]:
import numpy as np
from tqdm import tqdm

def get_embedding(text):
    inputs = bert_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = bert_model(**inputs)

    # Use CLS token instead of mean pooling
    embedding = outputs.last_hidden_state[:, 0, :]

    return embedding.cpu().numpy()

texts = df_all["text"][:20000]  # limit for speed

embeddings = []

for text in tqdm(texts):
    embeddings.append(get_embedding(text)[0])

embeddings = np.array(embeddings).astype("float32")

print("Embeddings ready")

100%|██████████| 20000/20000 [03:28<00:00, 95.79it/s]


Embeddings ready


Precompute Task-wise Index

In [ ]:
import faiss
faiss.normalize_L2(embeddings)

In [ ]:
task_indices = {}
task_texts = {}

for task in df_all["task"].unique():
    print(f"Processing task: {task}")

    task_df = df_all[df_all["task"] == task].reset_index(drop=True)

    #Remove noisy / very short texts
    task_df = task_df[task_df["text"].str.len() > 50]

    texts_subset = task_df["text"][:5000]

    embs = []

    for text in texts_subset:
        embs.append(get_embedding(text)[0])

    embs = np.array(embs).astype("float32")

    # Normalize for cosine similarity
    faiss.normalize_L2(embs)

    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)

    task_indices[task] = index
    task_texts[task] = texts_subset

print("All task indices ready ")

Processing task: quora
Processing task: admission
Processing task: stackoverflow
Processing task: trec
Processing task: coursera
All task indices ready 


Build FAISS Index

In [ ]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS index created")

FAISS index created


Retrieval Function

In [ ]:
def retrieve(query, top_k=3):
    task = route_query(query)

    index = task_indices[task]
    texts_subset = task_texts[task]

    query_embedding = get_embedding(query)
    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(query_embedding, top_k)

    results = [texts_subset.iloc[i] for i in indices[0]]

    return results

Load Fine-Tuned LLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/content/drive/MyDrive/tinyllama_realQA_lora"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path).to(device)

print("LLM loaded")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

LLM loaded


Build Prompt

In [ ]:
def build_prompt(query, contexts):
    context_text = "\n\n".join(contexts)

    prompt = f"""
You are a knowledgeable academic assistant.

Answer the question clearly and in 3-4 sentences.

Explain properly and give useful information.

Question:
{query}

Context:
{context_text}

Answer:
"""
    return prompt

Routing of query

In [ ]:
def route_query(query):
    query = query.lower()

    if any(x in query for x in ["gre", "toefl", "admission"]):
        return "admission"

    elif any(x in query for x in ["sql", "java", "exception", "code", "program"]):
        return "stackoverflow"

    elif any(x in query for x in ["course", "learn", "study"]):
        return "coursera"

    elif any(x in query for x in ["what is", "define", "explain"]):
        return "trec"

    else:
        return "quora"

In [ ]:
def rag_answer(query):
    contexts = retrieve(query)

    prompt = build_prompt(query, contexts)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2
    )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # remove prompt part
    if "Answer:" in full_output:
        answer = full_output.split("Answer:")[-1]
    else:
        answer = full_output

    return answer.strip()

RAG Answer Function

In [ ]:
def retrieve(query, top_k=3):
    task = route_query(query)

    # filter dataset
    filtered_df = df_all[df_all["task"] == task].reset_index(drop=True)

    filtered_texts = filtered_df["text"][:5000]  # limit for speed

    # create embeddings ONCE per task (simple approach)
    task_embeddings = []

    for text in filtered_texts:
        task_embeddings.append(get_embedding(text)[0])

    task_embeddings = np.array(task_embeddings).astype("float32")
    faiss.normalize_L2(task_embeddings)

    index = faiss.IndexFlatIP(task_embeddings.shape[1])
    index.add(task_embeddings)

    query_embedding = get_embedding(query)
    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(query_embedding, top_k)

    results = [filtered_texts.iloc[i] for i in indices[0]]

    return results

Format Display

In [ ]:
def display_query(query):
    print("\n" + "="*60)
    print("QUESTION:")
    print(textwrap.fill(query, width=60))
    print("-"*60)

    answer = rag_answer(query)

    print("ANSWER:\n")
    print(textwrap.fill(answer, width=80))
    print("="*60)

Test System

In [ ]:
display_query("What is SQL JOIN and how does it work?")



QUESTION:
What is SQL JOIN and how does it work?
------------------------------------------------------------
ANSWER:

SQL JOIN refers to a type of database query used by developers to join tables.
It involves joining two or more tables based on common fields, such as name,
age, etc. In this context, you're asking about a specific programming concept.


In [ ]:
display_query("Which course is best for learning data science?")


QUESTION:
Which course is best for learning data science?
------------------------------------------------------------
ANSWER:

The best course for learning data science would be the one with the highest
level of proficiency required by industry, which could range from undergraduate
to graduate courses depending on your career path. It's always good to research
different options before deciding.


In [ ]:
display_query("Are Coursera courses good for beginners?")



QUESTION:
Are Coursera courses good for beginners?
------------------------------------------------------------
ANSWER:

Yes, Coursera courses are generally considered good options for beginners due to
their convenience and accessibility. The courses cover topics relevant to
learning new skills or further developing existing ones.


In [ ]:
display_query("What is the difference between stack and queue?")


QUESTION:
What is the difference between stack and queue?
------------------------------------------------------------
ANSWER:

A stack is used to store data items in order, while a queue stores elements
dequeued from an underlying collection (such as list or array) simultaneously. A
stack can be viewed as an infinite linear heap of memory, whereas a queue
represents finite collections with fixed size. Stacks and queues are commonly
used in concurrent programming because they allow simultaneous access to data
stored in their respective structures without having to use synchronization
mechanisms such as locks.


**Results**


The system was tested on multiple academic queries across different domains such as programming, university admissions, and online courses. The model was able to retrieve relevant context using semantic search and generate meaningful responses using the fine-tuned language model.

The outputs demonstrate that the assistant can handle diverse queries and provide understandable answers. However, the quality of responses depends on the relevance of retrieved context and the limitations of the language model. In some cases, answers are general due to noisy or less descriptive data.

---

**System Architecture**

The system follows a Retrieval-Augmented Generation pipeline. User queries are first processed and routed to the appropriate domain. A BERT-based model converts the query into embeddings, which are used to retrieve relevant context from a FAISS index. The retrieved context is then passed to a fine-tuned language model (TinyLlama), which generates the final answer.

---

**Limitations**


The system has certain limitations. The retrieval component depends on the quality of the dataset, and noisy or short text entries can affect the relevance of results. Additionally, the language model used (TinyLlama) is relatively small, which limits its ability to generate highly detailed or precise answers.

The system also does not perform deep reasoning and may produce generic responses when context is insufficient. These limitations highlight the importance of high-quality data and larger models in RAG-based systems.

---

**Conclusion:**


In this project, we successfully developed a **College Assistant using Retrieval-Augmented Generation (RAG)** to answer academic and technical queries. The system combines a BERT-based retriever with a FAISS vector index to efficiently search through a multi-domain educational dataset, and a fine-tuned TinyLlama model to generate responses. By integrating retrieval with generation, the assistant is able to provide context-aware answers rather than relying solely on pretrained knowledge.

The system demonstrates the ability to handle diverse queries related to university admissions, programming, online courses, and general academic topics. Domain-based routing and semantic search improve the relevance of retrieved information, while prompt engineering enhances the clarity of generated responses. Although the quality of answers depends on the relevance of retrieved context and the limitations of a smaller language model, the overall pipeline effectively showcases the practical implementation of a RAG-based assistant.

In conclusion, this project highlights the effectiveness of combining retrieval and generation techniques for building intelligent assistants. Future improvements could include using larger language models, cleaner domain-specific datasets, and more advanced retrieval strategies to further enhance response accuracy and depth.
